Turn static reference images into a cinematic AI-generated video — scene by scene, with matching
durations and original audio. Pixeltable orchestrates the full pipeline in a **single table**: detect
scene boundaries, analyze each reference image with Gemini, build dynamic prompts, generate clips with
Veo 3, trim to duration, and assemble the final video with `concat_videos_agg` — all declaratively.

## Problem

You have a source video (e.g., a real estate walkthrough) and a set of reference images — one per scene.
You want to regenerate the video with Veo 3 so that each still image becomes a living, cinematic
3D scene with animated elements, matching the original scene durations and audio.

Doing this manually means: detecting scene boundaries, analyzing each image, writing prompts,
calling the generation API per scene, stitching clips, and mixing audio.

| Use case | Input | Output |
|----------|-------|--------|
| Real estate | Property photos + walkthrough clip | Cinematic AI-animated property video |
| Automotive | Car stills + promo video | AI-animated car commercial |
| Hospitality | Hotel photos + tour video | Immersive virtual tour |
| Architecture | Renders + flythrough clip | AI-animated architectural flythrough |

## Solution

**What Pixeltable handles — in one table:**

- **Scene detection.** Find scene boundaries in the source clip (local, no API cost) to get timing metadata.
- **Image analysis.** A single Gemini call per image extracts content (subject, environment, mood) and
  inferred camera style (lens type, camera movement, framing).
- **Dynamic prompts.** A UDF builds a cinematic Veo 3 prompt with animation directives tailored to each scene.
- **Video generation.** Veo 3 turns each still into a living, cinematic clip.
- **Trim.** Each clip is trimmed to match the exact scene duration.
- **Assembly.** `concat_videos_agg` aggregates clips directly from the scenes table. Two versions:
  trimmed (matching original scene durations) and full (8-second clips). Both get the original audio.
- **Social media crops.** Center-cropped to 9:16, 1:1, and 16:9.

```
Source Clip ──▶ Scene Detection (Python) ──▶ scene durations
    │                                              │
    └──▶ Extract Audio                             │
              │                                    ▼
Reference Images + durations ──▶ scenes table (single table)
                                    │
                                    ├─▶ Gemini 2.5 Flash (content + camera)
                                    ├─▶ build_veo_prompt() (dynamic prompt)
                                    ├─▶ Veo 3 (generate clip)
                                    ├─▶ clip() (trim to exact duration)
                                    │
                                    ├─▶ concat_videos_agg() ──▶ trimmed movie
                                    │         + original audio ──▶ trimmed_final
                                    │
                                    ├─▶ concat_videos_agg() ──▶ full movie (8s clips)
                                    │         + original audio ──▶ full_final
                                    │
                                    │
                                    └─▶ center crop ──▶ reels / square / landscape
```

| Step | Model / Tool | Purpose |
|------|-------------|--------|
| Scene detect | Built-in (PySceneDetect) | Extract scene durations from source clip |
| Image analysis | Gemini 2.5 Flash | Content + camera style from each reference image |
| Prompt | Custom UDF | Build cinematic 3D animation prompt per scene |
| Generate | Veo 3 | Turn still image into living cinematic clip |
| Trim | Built-in (ffmpeg) | Trim clip to exact scene duration |
| Concatenate | `concat_videos_agg` (ffmpeg) | Aggregate clips from the scenes table |
| Extract audio | Built-in (PyAV) | Pull audio track from source clip |
| Combine | Built-in (ffmpeg) | Overlay original audio onto video |
| Crop | Built-in (ffmpeg) | Center-crop to social media aspect ratios |

**API keys needed:** Gemini (Gemini 2.5 Flash + Veo 3). Scene detection runs locally.

**Tables:** One main table (`veo_demo.scenes`) + one assembly table (`veo_demo.output`).

### Setup

In [ ]:
%pip install -qU pixeltable google-genai scenedetect

In [ ]:
import getpass
import os

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Google AI Studio API Key: ')



In [ ]:
import json

import pixeltable as pxt
from pixeltable.functions import gemini
from pixeltable.functions.video import (
    clip,
    concat_videos_agg,
    crop,
    extract_audio,
    with_audio,
)

pxt.drop_dir('veo_demo', force=True)
pxt.create_dir('veo_demo')

In [ ]:
import pathlib

NOTEBOOK_DIR = pathlib.Path().resolve()
RESOURCES_DIR = NOTEBOOK_DIR / 'resources'

CLIP_PATH = str(RESOURCES_DIR / 'clip_2.mp4')

IMAGE_FILES = [
    'capture.png',
    'capture_1.png',
    'capture_2.png',
    'capture_3.png',
]

## Build the Pipeline

### Step 1: Detect Scene Durations

Scene detection is a one-shot operation — no need for a dedicated table. We run it directly
in Python to extract timing metadata (duration per scene) that will feed into the scenes table.

In [ ]:
from scenedetect import detect, ContentDetector

scene_boundaries = detect(CLIP_PATH, ContentDetector())
scene_list = [
    {'start_time': s[0].get_seconds(), 'duration': (s[1] - s[0]).get_seconds()}
    for s in scene_boundaries
]

num_scenes = min(len(IMAGE_FILES), len(scene_list))
print(f'Detected {len(scene_list)} scenes, using {num_scenes} (matched to {len(IMAGE_FILES)} images)')
for i, s in enumerate(scene_list[:num_scenes]):
    print(f"  Scene {i + 1}: duration={s['duration']:.2f}s")

### Step 2: Create the Scenes Table

This is the **only table** in the pipeline. Each row pairs a reference image with its target
scene duration. All AI analysis, generation, and trimming run as computed columns on this table.

In [ ]:
scenes = pxt.create_table(
    'veo_demo.scenes',
    {
        'image': pxt.Image,
        'scene_order': pxt.Int,
        'scene_duration': pxt.Float,
    },
)

### Step 3: Analyze Each Image

A single Gemini call per image extracts everything we need for the prompt: the **content**
(subject, environment, mood) and the inferred **camera style** (lens type, camera movement, framing)
— all from the reference image itself.

In [ ]:
ANALYSIS_PROMPT = (
    'Analyze this image from a luxury property listing. Provide:\n'
    '1. Subject: the main focal point or feature.\n'
    '2. Environment: the room, space, or setting.\n'
    '3. Mood: the overall atmosphere.\n'
    '4. Lens type: infer from perspective (wide-angle, standard, telephoto).\n'
    '5. Camera movement: suggest the best cinematic movement to animate this scene '
    '(slow dolly forward, gentle pan, drone pullback, steadicam walkthrough, static with parallax).\n'
    '6. Framing: the composition style (wide establishing, medium, close-up, detail).\n'
    'Be specific and concise.'
)

scenes.add_computed_column(
    analysis=gemini.generate_content(
        [scenes.image, ANALYSIS_PROMPT],
        model='gemini-2.5-flash',
        config={
            'response_mime_type': 'application/json',
            'response_schema': {
                'type': 'OBJECT',
                'properties': {
                    'subject': {'type': 'STRING'},
                    'environment': {'type': 'STRING'},
                    'mood': {'type': 'STRING'},
                    'lens_type': {'type': 'STRING'},
                    'camera_movement': {'type': 'STRING'},
                    'framing': {'type': 'STRING'},
                },
                'required': ['subject', 'environment', 'mood', 'lens_type', 'camera_movement', 'framing'],
            },
        },
    )
)

### Step 4: Build Cinematic Prompts

A UDF assembles the Veo 3 prompt from the image analysis and scene duration. The prompt directs
Veo 3 to render each still as a **living 3D cinematic scene** — with slow-motion effects, animated
environmental elements (trees swaying, water rippling, light shifting), depth, and immersion.

In [ ]:
@pxt.udf
def build_veo_prompt(analysis: dict, scene_duration: float) -> str:
    """Build a cinematic Veo 3 prompt with scene-aware animation."""
    a = json.loads(analysis['candidates'][0]['content']['parts'][0]['text'])
    subject = a["subject"]
    env = a["environment"]
    mood = a["mood"]
    env_lower = env.lower()

    animations = []
    if any(w in env_lower for w in ['pool', 'water', 'ocean', 'lake', 'fountain']):
        animations.append('water ripples with light caustics dancing on the surface')
    if any(w in env_lower for w in ['garden', 'yard', 'exterior', 'outdoor', 'lawn', 'landscape']):
        animations.append('trees sway and leaves flutter in the wind, grass ripples gently')
    if any(w in env_lower for w in ['bedroom', 'living', 'interior', 'lounge', 'suite']):
        animations.append('curtains billow softly in a breeze, light shifts across surfaces')
    if any(w in env_lower for w in ['kitchen', 'dining', 'bathroom', 'closet', 'dressing']):
        animations.append('reflections shimmer on polished surfaces and glass')
    if any(w in env_lower for w in ['exterior', 'facade', 'aerial', 'drone', 'overview']):
        animations.append('clouds drift across the sky, shadows move slowly across the ground')
    if any(w in env_lower for w in ['fireplace', 'hearth']):
        animations.append('fireplace flames flicker warmly, casting dancing shadows')
    if not animations:
        animations.append('sunlight shifts and casts moving golden rays, dust motes float through light')

    anim_text = '. '.join(animations)

    return (
        f"Cinematic luxury real estate video of {subject} "
        f"in {env}. Mood: {mood}. "
        f"Camera: {a['lens_type']} lens, {a['camera_movement']}. "
        f"Framing: {a['framing']}. "
        f"Bring this scene to life as a cinematic 3D world in motion. "
        f"{anim_text}. "
        f"Everything feels alive with natural depth and immersion. "
        f"Cinematic slow-motion, smooth parallax camera depth, "
        f"volumetric lighting, shallow depth of field. "
        f"Photorealistic, 8K quality, professional cinematography."
    )


scenes.add_computed_column(
    dynamic_prompt=build_veo_prompt(scenes.analysis, scenes.scene_duration)
)

### Step 5: Generate Clips with Veo 3

Each scene gets a cinematic video clip from Veo 3, driven by the dynamic prompt **and** the reference
image as a visual anchor. The `duration_seconds` config tells Veo 3 the target length for each clip.
After generation, `clip()` trims each result to the exact scene duration as a safety net.

In [ ]:
scenes.add_computed_column(
    generated_clip=gemini.generate_videos(
        prompt=scenes.dynamic_prompt,
        image=scenes.image,
        model='veo-3.0-generate-001',
    )
)

Trim each generated clip to match the exact scene duration from the source video:

In [ ]:
scenes.add_computed_column(
    trimmed_clip=clip(
        scenes.generated_clip,
        start_time=0.0,
        duration=scenes.scene_duration,
    )
)

### Step 6: Insert Reference Images

Each image is paired with its scene duration from Step 1. Inserting triggers the full computed
column cascade: image analysis → dynamic prompt → Veo 3 → trim.

In [ ]:
rows = [
    {
        'image': str(RESOURCES_DIR / IMAGE_FILES[i]),
        'scene_order': i + 1,
        'scene_duration': scene_list[i]['duration'],
    }
    for i in range(num_scenes)
]

scenes.insert(rows)

Inspect the generated prompts — each one combines content, camera style, and cinematic animation directives:

In [ ]:
scenes.order_by(scenes.scene_order).select(
    scenes.scene_order,
    scenes.dynamic_prompt,
).collect()

Preview the reference images alongside their generated cinematic clips:

In [ ]:
scenes.order_by(scenes.scene_order).select(
    scenes.image,
    scenes.generated_clip,
    scenes.trimmed_clip,
).collect()

## Assemble the Final Video

### Step 7: Concatenate and Combine

`concat_videos_agg` aggregates clips directly from the scenes table — no intermediate collect/insert.
Two video versions are assembled:

1. **Trimmed + original audio** — clips cut to match original scene durations, with the source audio.
2. **Full 8s clips + original audio** — uncut generated clips with the source audio.

All versions get center-cropped to social media formats.

In [ ]:
@pxt.udf
def center_crop_box(video: pxt.Video, ratio_w: int, ratio_h: int) -> list:
    """Return a center crop [x, y, w, h] for a target aspect ratio."""
    import av
    with av.open(video) as container:
        stream = container.streams.video[0]
        src_w, src_h = stream.width, stream.height
    target = ratio_w / ratio_h
    if src_w / src_h > target:
        w = int(src_h * target)
        h = src_h
    else:
        w = src_w
        h = int(src_w / target)
    return [(src_w - w) // 2, (src_h - h) // 2, w, h]


trimmed_movie = scenes.select(
    concat_videos_agg(scenes.scene_order, scenes.trimmed_clip)
).collect()[0]['concat_videos_agg']

full_movie = scenes.select(
    concat_videos_agg(scenes.scene_order, scenes.generated_clip)
).collect()[0]['concat_videos_agg']

Assemble both versions — trimmed with original audio, full with original audio — plus social media crops:

In [ ]:
output = pxt.create_table('veo_demo.output', {
    'trimmed_movie': pxt.Video,
    'full_movie': pxt.Video,
    'source_video': pxt.Video,
})

output.add_computed_column(audio=extract_audio(output.source_video, format='mp3'))
output.add_computed_column(trimmed_final=with_audio(output.trimmed_movie, output.audio))
output.add_computed_column(full_final=with_audio(output.full_movie, output.audio))

output.add_computed_column(
    reels_vertical=crop(output.trimmed_final, center_crop_box(output.trimmed_final, 9, 16))
)
output.add_computed_column(
    feed_square=crop(output.trimmed_final, center_crop_box(output.trimmed_final, 1, 1))
)
output.add_computed_column(
    youtube_landscape=crop(output.trimmed_final, center_crop_box(output.trimmed_final, 16, 9))
)

output.insert([{
    'trimmed_movie': trimmed_movie,
    'full_movie': full_movie,
    'source_video': CLIP_PATH,
}])

Compare both versions — original audio, full with original audio:

In [ ]:
output.select(
    output.trimmed_final,
    output.full_final,
).collect()

Social media crops (9:16 Reels, 1:1 Feed, 16:9 YouTube):

In [ ]:
output.select(
    output.reels_vertical,
    output.feed_square,
    output.youtube_landscape,
).collect()

## What's Next

### Tune Scene Detection

Adjust the threshold to control how many scenes are detected:

```python
from scenedetect import detect, ContentDetector
scenes = detect(clip_path, ContentDetector(threshold=15.0))
```

### Customize the Animation Style

Edit `build_veo_prompt` to change the creative direction — the animation directives are already
conditional on the scene content, so adding new keywords and effects is straightforward.

### Add Captions

Use `overlay_text` to add room labels or property titles per scene:

```python
scenes.add_computed_column(
    captioned=scenes.trimmed_clip.overlay_text(
        text='Master Suite',
        font_size=48,
        color='white',
        box=True,
        box_color='black',
        box_opacity=0.6,
        vertical_align='bottom',
        vertical_margin=40,
    )
)
```

### Explore More

- [AI Video Ad Generator](https://docs.pixeltable.com/howto/cookbooks/video/video-ad-generator) — full ad pipeline with Whisper, scene detection, and Veo 3
- [Scene Detection](https://docs.pixeltable.com/howto/cookbooks/video/video-scene-detection) — all five built-in scene detection algorithms
- [Extract Audio from Video](https://docs.pixeltable.com/howto/cookbooks/audio/audio-extract-from-video) — audio extraction patterns
- [Table as UDF](https://docs.pixeltable.com/howto/cookbooks/agents/pattern-table-as-udf) — wrap a table pipeline as a reusable function